# Multiple Linear Regression

**Objective:** Demonstrate multiple linear regression using two predictors

**Author:** Dr. Juan Manuel Ahuactzin Larios

## Regression with the statsmodels library  
https://www.statsmodels.org/

In [5]:
# Data libraries
import pandas as pd
import numpy as np

# Graph
import matplotlib.pyplot as plt
from matplotlib import style
import plotly.graph_objects as go

# Processing and models
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

#Statistics
from scipy.stats import pearsonr

import statsmodels.api as sm

In [6]:
plt.rcParams['image.cmap'] = "bwr"
plt.rcParams['figure.dpi'] = "100"
plt.rcParams['savefig.bbox'] = "tight"
style.use('ggplot') or plt.style.use('ggplot')

# warning configuration
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

In [7]:
def plot_regression_plane(model, data, X, predictions):
    # Coefficients of the model
    w0 = model.params[0]
    w1 = model.params[1]    
    w2 = model.params[2]

    # Create a grid of values for x1 and x2
    x1_min = data['Poblacion'].min()
    x1_max = data['Poblacion'].max()
    x2_min = data['Confirmados'].min()
    x2_max = data['Confirmados'].max()

    x1_range = np.linspace(x1_min, x1_max, 50)
    x2_range = np.linspace(x2_min, x2_max, 50)

    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)

    # This is the equation of the plane defined by the linear regression model
    z = w0 + w1*x1_grid + w2*x2_grid

    # Figure
    fig = go.Figure()

    # Display the plane
    fig.add_trace(go.Surface(
        x=x1_grid,
        y=x2_grid,
        z=z,
        colorscale='Viridis',
        opacity=0.8,
        hovertemplate=
            "Población: %{x:,}<br>" +
            "Confirmados: %{y:,}<br>" +
            "Defunciones (pred): %{z:,.0f}<extra></extra>"
    ))

    # Display real data
    fig.add_trace(go.Scatter3d(
        x=X.values[:,1],
        y=X.values[:,2],
        z=predictions["y_true"],
        mode='markers',
        marker=dict(color='red', size=5),
        name='True value',
        text=[f"{v:,.0f}" for v in predictions["y_true"]],
        customdata=X.index, 
        hovertemplate=
            "Estado: %{customdata}<br>" +
            "Población: %{x:,}<br>" +
            "Confirmados: %{y:,}<br>" +
            "Defunciones (real): %{text}<extra></extra>"
    ))

    # Display predictions
    fig.add_trace(go.Scatter3d(
        x=X.values[:,1],
        y=X.values[:,2],
        z=predictions["mean"],
        mode='markers',
        marker=dict(color='blue', size=5),
        name='Prediction',
        text=[f"{v:,.0f}" for v in predictions["mean"]],
        customdata=X.index, 
        hovertemplate=
            "Estado: %{customdata}<br>" +
            "Población: %{x:,}<br>" +
            "Confirmados: %{y:,}<br>" +
            "Defunciones (pred): %{text}<extra></extra>"
    ))

    # Set layout with titles and axis labels
    fig.update_layout(
        title='Linear Regression Plane',
        width=900,
        height=700,
        scene=dict(
            xaxis_title='Población',
            yaxis_title='Confirmados',
            zaxis_title='Defunciones',
            aspectmode='cube',
            camera=dict(
            eye=dict(x=0.9021, y=-2.02, z=1.19)  # 👈 posición de la cámara
            )
        )
    )

    fig.show()

## Matplotlib and Warnings Configuration

## Data Loading

In [8]:
data = pd.read_excel("../data/Datos_COVID_por_estado.xlsx", index_col = 0) 

## Data Visualization

In [9]:
fig = go.Figure()


# Scatter (puntos)
fig.add_trace(go.Scatter3d(
    x=data['Poblacion'],
    y=data['Confirmados'],
    z=data['Defunciones'],
    mode='markers',
    marker=dict(color='red', size=7),
    name='Data',
    text=[f"y: {v:,.0f}" for v in data['Defunciones']],
    customdata=data.index, 
    hovertemplate=
            "Estado: %{customdata}<br>" +
            "Población: %{x:,}<br>" +
            "Confirmados: %{y:,}<br>" +
            "Defunciones: %{text}<extra></extra>"
    ))

fig.update_layout(
    title='Linear Regression Plane',
    width=900,
    height=700,
    scene=dict(
        xaxis_title='Población',
        yaxis_title='Confirmados',
        zaxis_title='Defunciones',
        aspectmode='cube',
        camera=dict(
            eye=dict(x=0.9021, y=-2.02, z=1.19)  # 👈 posición de la cámara
        )
    )
)

fig.show()

## Running the Linear Regression

In [10]:
# Define dedictors and target variable
X = data[['Poblacion', 'Confirmados']]
y = data[['Defunciones']]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=1966,
    shuffle=True
)

# Add constant term to the predictors for the intercept
X_train = sm.add_constant(X_train, prepend=True)

# Costruction of the model
model = sm.OLS(endog=y_train, exog=X_train)
trained_model = model.fit()

## Printing the Results of the Linear Regression Model

In [11]:
print(trained_model.summary())
print()
print(f"Our model is by consequence: ")
print(f"{y.columns[0]} = {trained_model.params[0]} +",end='')
for i in range(1, len(trained_model.params)-1):
    print(f" {trained_model.params[i]}*{X.columns[i-1]} + ", end='')
print(f"{trained_model.params[len(trained_model.params)-1]}*{X.columns[len(trained_model.params)-2]}", end='')


                            OLS Regression Results                            
Dep. Variable:            Defunciones   R-squared:                       0.953
Model:                            OLS   Adj. R-squared:                  0.949
Method:                 Least Squares   F-statistic:                     222.7
Date:                Fri, 20 Mar 2026   Prob (F-statistic):           2.51e-15
Time:                        09:22:58   Log-Likelihood:                -216.92
No. Observations:                  25   AIC:                             439.8
Df Residuals:                      22   BIC:                             443.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          64.2473    594.503      0.108      

## Prediction and Calculation of the Standard Error of the Residuals

In [12]:
# Prediction with an confidence inteval of 95%
predictions = trained_model.get_prediction(exog = X_train).summary_frame(alpha=0.05)
predictions['y_true'] = y_train

# Display mea, mean standar error, mean confidence interval lower, mean confidence interval upper, 
# observation confidence interval lower, observation confidence interval upper 
display(predictions.head(4))

for i, p in enumerate(predictions["mean"]):
    print(f" True value vs predicted: {y_train.iloc[i,0]} {int(p)} ")
  
rmse = root_mean_squared_error(
        y_true  = y_train,
        y_pred  = predictions["mean"]
       )
print("")
print(f"Root Mean Square Error : {rmse}")

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper,y_true
Estado,,,,,,,
MORELOS,3448.339932,365.062425,2691.246801,4205.433063,220.862554,6675.817309,3575
TLAXCALA,2234.111842,426.971383,1348.627391,3119.596293,-1025.873269,5494.096953,2563
DISTRITO FEDERAL,35048.283234,1488.225801,31961.891826,38134.674641,30647.234253,39449.332215,34674
CHIHUAHUA,6122.687389,319.778629,5459.507103,6785.867675,2915.940080,9329.434698,7553


 True value vs predicted: 3575 3448 
 True value vs predicted: 2563 2234 
 True value vs predicted: 34674 35048 
 True value vs predicted: 7553 6122 
 True value vs predicted: 3015 3053 
 True value vs predicted: 1666 6376 
 True value vs predicted: 2498 3282 
 True value vs predicted: 8728 5686 
 True value vs predicted: 4468 5022 
 True value vs predicted: 5230 6252 
 True value vs predicted: 2469 2538 
 True value vs predicted: 4557 5375 
 True value vs predicted: 6775 6199 
 True value vs predicted: 5977 6847 
 True value vs predicted: 2836 2938 
 True value vs predicted: 1529 2366 
 True value vs predicted: 6405 4885 
 True value vs predicted: 6285 4707 
 True value vs predicted: 12681 11997 
 True value vs predicted: 4205 4125 
 True value vs predicted: 11114 11437 
 True value vs predicted: 1306 1526 
 True value vs predicted: 4276 5560 
 True value vs predicted: 6414 5974 
 True value vs predicted: 12271 10061 

Root Mean Square Error : 1419.1626639879232


## Visualization of the Predictions

In [13]:
plot_regression_plane(trained_model, data, X_train, predictions)

## Prediction with Test Data

In [14]:
print(f"X_test shape: {X_test.shape}")
X_test = sm.add_constant(X_test, prepend=True)
predictions_t = trained_model.get_prediction(exog = X_test).summary_frame(alpha=0.05)

predictions_t['y_true'] = y_test

print(f"X_test shape: {X_test.shape}")
print(f"Len predictions: {len(predictions_t)}")
print(f"Len y_test: {len(y_test)}")

for i, p in enumerate(predictions_t["mean"]):
    print(f" True value vs predicted: {y_test.iloc[i,0]} {int(p)} ")
  
rmse = root_mean_squared_error(
        y_true  = y_test,
        y_pred  = predictions_t["mean"]
       )

print("")
print(f"Root Mean Square Error : {rmse}")

X_test shape: (7, 2)
X_test shape: (7, 3)
Len predictions: 7
Len y_test: 7
 True value vs predicted: 10139 11236 
 True value vs predicted: 5443 5459 
 True value vs predicted: 9738 10586 
 True value vs predicted: 1221 1331 
 True value vs predicted: 37359 27621 
 True value vs predicted: 1873 1858 
 True value vs predicted: 3960 6123 

Root Mean Square Error : 3806.7760872666167


## Visualization of the Predictions

In [15]:
plot_regression_plane(trained_model, data, X_test, predictions_t)